In [1]:
#fill NA with None
from sklearn import tree
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import random
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold
from statistics import mean
from sklearn.naive_bayes import CategoricalNB


In [2]:
#https://stackoverflow.com/questions/20224526/how-to-extract-the-decision-rules-from-scikit-learn-decision-tree
def get_code(tree, feature_names):
        left      = tree.tree_.children_left
        right     = tree.tree_.children_right
        threshold = tree.tree_.threshold
        features  = [feature_names[i] for i in tree.tree_.feature]
        value = tree.tree_.value

        def recurse(left, right, threshold, features, node):
                if (threshold[node] != -2):
                        print("if ( " + features[node] + " <= " + str(threshold[node]) + " ) {")
                        if left[node] != -1:
                                recurse (left, right, threshold, features,left[node])
                        print( "} else {")
                        if right[node] != -1:
                                recurse (left, right, threshold, features,right[node])
                        print( "}")
                else:
                        print( "return " + str(value[node]))

        recurse(left, right, threshold, features, 0)

In [3]:
# data = pd.read_excel('dataset_all_original.xlsx') 
# data = pd.read_excel('dataset_all_gen_ssdp.xlsx') 
# data = pd.read_excel('dataset_all_gen.xlsx') 
data = pd.read_excel('dataset_all_original_ssdp.xlsx') 
# data = pd.read_excel('dataset_all_original_add_nobias.xlsx') 

In [4]:
data = data.sample(frac = 1) 

In [5]:

data = data.reset_index(drop=True)

In [6]:
data.isnull().sum()

name             0
type             0
vendor           0
DNS           2491
mDNS           937
nupnp        11636
open_port      875
ssdp_type     1361
ssdp_t        1361
ssdp_v        1361
dtype: int64

In [7]:
data = data.fillna("NaN")

In [8]:
data.isnull().sum()

name         0
type         0
vendor       0
DNS          0
mDNS         0
nupnp        0
open_port    0
ssdp_type    0
ssdp_t       0
ssdp_v       0
dtype: int64

In [9]:
data.dtypes

name         object
type         object
vendor       object
DNS          object
mDNS         object
nupnp        object
open_port    object
ssdp_type    object
ssdp_t       object
ssdp_v       object
dtype: object

In [10]:
data = data.astype('category')

In [11]:
data.dtypes

name         category
type         category
vendor       category
DNS          category
mDNS         category
nupnp        category
open_port    category
ssdp_type    category
ssdp_t       category
ssdp_v       category
dtype: object

In [12]:
data

,name,type,vendor,DNS,mDNS,nupnp,open_port,ssdp_type,ssdp_t,ssdp_v
0,Withings Smart scale,Health-Monitor,Withings,NaN,_solarbreeze._tcp.local.,NaN,NaN,NOTIFY * HTTP/1.1,LOCATION,http://192.168.0.102:80/description.xml
1,Apple iPhone,Non-IOT,"Apple, Inc.",marketwatch.com,_bytecard._tcp.local.,NaN,tcp/1900,M-SEARCH * HTTP/1.1,MX,1
2,Blipcare Blood Pressure meter,Health-Monitor,Blipcare,NaN,_temp._udp.local.,NaN,tcp/80,NOTIFY * HTTP/1.1,OPT,http://schemas.upnp.org/upnp/1/0/;
3,Google OnHub,Hubs,Google,bloglovin.com,_toughjoyfax._tcp.local.,NaN,tcp/1883,NOTIFY * HTTP/1.1,NT,urn:schemas-upnp-org:service:WANPPPConnection:1
4,Belkin wemo motion sensor,Energy Management,Belkin International Inc.,wikispaces.com,_tin._tcp.local.,NaN,tcp/443,NOTIFY * HTTP/1.1,01-NLS,1
5,samsung smart home camera,Cameras,SAMSUNG,ow.ly,_biodex._udp.local.,NaN,tcp/1883,NOTIFY * HTTP/1.1,NTS,ssdp:alive
6,samsung smart home camera,Cameras,SAMSUNG,whitehouse.gov,_asoka._tcp.local.,NaN,tcp/443,NOTIFY * HTTP/1.1,CACHE-CONTROL,max-age=100
7,Philips Hue Bridge,Hubs,Philips Lighting BV,webmd.com,_toughjoyfax._tcp.local.,None,tcp/1883,NOTIFY * HTTP/1.1,ST,upnp:rootdevice
8,LiFX,Energy Management,LIFI LABS MANAGEMENT PTY LTD,"('v2.broker.lifx.co', 'A')",NaN,NaN,NaN,NaN,NaN,NaN
9,Apple HomePod,Hubs,"Apple, Inc.",homestead.com,_otcom._udp.local.,NaN,tcp/80,NOTIFY * HTTP/1.1,Server,SDK 4.3.0.0 UPnP/1.0 MiniUPnPd/1.6


In [13]:
y = data.drop_duplicates(keep='first')
# y = y.drop(['name'], axis=1)
# y = y.drop(['Name'], axis=1)
y = y.drop_duplicates(keep='first')
# y = y.reset_index()
# shuffle the DataFrame rows
# y = y.sample(frac = 1) 


In [14]:
y = y.astype(str)
len(y)

12000

In [15]:
y = y.apply(LabelEncoder().fit_transform)


In [16]:
list_data = []
list_data.append(y)
len(list_data)

1

In [17]:
accuracy = []
accuracy_t = []
acc = []

In [18]:
for data in list_data:
    for i in range(10):
        accuracy_temp = []
        kf = KFold(n_splits=10, random_state=None, shuffle=True)
#         kf.get_n_splits(data)
        for train_index, test_index in kf.split(data):
        #     print(index)
#             print("TRAIN:", train_index, "TEST:", test_index)
            train, test = data.iloc[train_index], data.iloc[test_index]
#             print("TRAIN:", train, "TEST:", test)
        #     print(len(train))
#             print("TRAIN:", len(train), "TEST:", len(test))
            label_data = train.type
            train_data = train.drop(['type'], axis=1)
            tlabel_data = test.type
            test_data = test.drop(['type'], axis=1)
            truelabel_data = pd.DataFrame(tlabel_data)
            truelabel_data = truelabel_data.reset_index(drop=True)
            clf = CategoricalNB()
#             clf = clf.partial_fit(train_data,label_data,[0,1,2,3,4,5] )
            clf = clf.fit(train_data,label_data )
            predict = clf.predict(test_data)
            result_label_data = pd.DataFrame(columns = ['type'])
            result_label_data.type = predict
            compare = pd.DataFrame(columns = ['truelabel_data','predictedlabel'])
            compare.truelabel_data = truelabel_data.type
            compare.predictedlabel = result_label_data.type
#             print(compare)
            t = (accuracy_score(compare.truelabel_data, compare.predictedlabel))
            accuracy_temp.append(t)
            acc.append(t)
            print(i,"  Accuracy: ",t)
        accuracy_t.append(mean(accuracy_temp))
    accuracy.append(mean(accuracy_t))

0   Accuracy:  1.0
0   Accuracy:  0.9991666666666666
0   Accuracy:  0.9966666666666667
0   Accuracy:  1.0
0   Accuracy:  1.0
0   Accuracy:  0.9991666666666666
0   Accuracy:  0.9991666666666666
0   Accuracy:  0.9983333333333333
0   Accuracy:  0.9991666666666666
0   Accuracy:  0.9975
1   Accuracy:  0.9991666666666666
1   Accuracy:  0.9991666666666666
1   Accuracy:  1.0
1   Accuracy:  1.0
1   Accuracy:  0.9975
1   Accuracy:  0.9983333333333333
1   Accuracy:  0.9966666666666667
1   Accuracy:  1.0
1   Accuracy:  1.0
1   Accuracy:  1.0
2   Accuracy:  1.0
2   Accuracy:  0.9983333333333333
2   Accuracy:  0.9983333333333333
2   Accuracy:  0.9975
2   Accuracy:  0.9991666666666666
2   Accuracy:  0.9991666666666666
2   Accuracy:  1.0
2   Accuracy:  1.0
2   Accuracy:  1.0
2   Accuracy:  1.0
3   Accuracy:  1.0
3   Accuracy:  1.0
3   Accuracy:  1.0
3   Accuracy:  0.9983333333333333
3   Accuracy:  0.9991666666666666
3   Accuracy:  0.9983333333333333
3   Accuracy:  0.9991666666666666
3   Accuracy:  0.9

In [19]:
print(len(y),"\t Accuracy for dataset_gen: \t", accuracy[0])
# print(len(dataset_all_gen_y),"\t Accuracy for dataset_all_gen: \t", accuracy[1])
# print(len(dataset_all_original_y),"\t Accuracy for dataset_all_original: \t", accuracy[2])
# print(len(dataset_jaccard_gen_y),"\t Accuracy for dataset_jaccard_gen: \t", accuracy[3])
# print(len(dataset_jaccard_original_y),"\t Accuracy for dataset_jaccard_original: \t", accuracy[4])
# print(len(dataset_myroom_gen_y),"\t Accuracy for dataset_myroom_gen: \t", accuracy[5])
# print(len(dataset_myroom_original_y),"\t Accuracy for dataset_myroom_original: \t", accuracy[6])
# print(len(dataset_university_gen_y),"\t Accuracy for dataset_university_gen: \t", accuracy[7])
# print(len(dataset_university_original_y),"\t Accuracy for dataset_university_original: \t", accuracy[8])

12000 	 Accuracy for dataset_gen: 	 0.9990083333333333


In [20]:
print(len(y),"\t STD for dataset_gen: \t", np.std(acc))

12000 	 STD for dataset_gen: 	 0.0009696434052440691
